## Games Management System


In [ ]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
from ipywidgets import Layout

# this wasnt built in with the standard module but is still ipywidgets
#only lets me wipe the screen by typing clear_output()
from IPython.display import clear_output

#Upload the pyc files
from google.colab import files
uploaded = files.upload()
import subscriptionManager as smSL
import feedbackManager as fmSL


Saving subscriptionManager.pyc to subscriptionManager (2).pyc
Saving feedbackManager.pyc to feedbackManager (2).pyc


Global variables

- selectedRow: for highlighting a row in the database.
- videoTab/boardTab: so the children of the Database can be changed to reflect a search term.
- rentalGraph gets updated every time a button is selected which can error if its respective widget is not instantiated
- screenBox: the main output box can change from inside multiple functions
- radio: not sure why this made an error but its for the radiobox to be able to change inside its observe function
- leftBox/rightBox: the two sub-boxes from screenbox. Only used to prevent rentalGraph accidentally rendering

In [ ]:
global selectedRow, videoTab, boardTab, rentalGraph, screenBox, radio
global leftBox, rightBox

## Groups
This first section defines most of the building blocks of the code. I've named them groups. They are merely collections of widgets that form part of the menu screen. I grouped them together for easy readability and modification. The coursework specific labelling is also here, ie gameSearch, gameRent, which is similar enough to grouping the other widgets


In [ ]:
# @title clear

# function used in most groups to clear text fields
def clear(btn):
  for widget in btn.linked_texts:
        widget.value = ""


In [ ]:
# @title delete group
def submitFunctionDelete(btn):
  GameID = GameIDDelete.value
  done = False

  currentV = search('all','Video_Game_Info.txt')
  currentB = search('all','Board_Game_Info.txt')

  vTitles = currentV[0]
  vData = currentV[1:]

  # Grab data as a list, delete specific row from list, then write back
  # GameID could be for board game or video game so check both

  newVidRows = []
  for item in vData:
    if item[0] == GameID and done == False:
      currentV.remove(item)
      successLabelDelete.value = f'Successfully Deleted {item[1]}'
      done = True
    else:
      newVidRows.append(item)

  if done == True:
    Vfile = open('Video_Game_Info.txt', 'w')
    stringTitlesV = ",".join(vTitles) + "\n"
    Vfile.write(stringTitlesV)
    for i in range(len(newVidRows)):
      Vfile.write(','.join(newVidRows[i]))
      if i != len(newVidRows) - 1: # guard against empty file lines
        Vfile.write("\n")

    Vfile.close()
    refreshDatabase()
    return

  bTitles = currentB[0]
  bData = currentB[1:]

  newBoardRows = []

  for item in bData:
    if item[0] == GameID and done == False:
      currentB.remove(item)
      successLabelDelete.value = f'Successfully Deleted {item[1]}'
      done = True
    else:
      newBoardRows.append(item)

  if done == True:
    Bfile = open('Board_Game_Info.txt', 'w')
    stringTitlesB = ",".join(bTitles) + "\n"
    Bfile.write(stringTitlesB)
    for i in range(len(newBoardRows)):
      Bfile.write(','.join(newBoardRows[i]))
      if i != len(newBoardRows) - 1:
        Bfile.write("\n")

    Bfile.close()
    refreshDatabase()
    return

  if done == False:
    successLabelDelete.value = f'Unable to delete {GameIDDelete.value}'
  refreshDatabase()
  return

#Widgets that build the delete area
deleteGame = widgets.Label(value = 'Delete game')
GameIDDelete = widgets.Text(description = 'Game ID:',
                            layout = Layout(width = '150px'))

#emptyLabel only exists for alignment purposes
EmptyLabel = widgets.Label(value = '', layout = Layout(width = '150px'))
submitBtnDelete = widgets.Button(description = 'Delete')
clearBtnDelete = widgets.Button(description = 'Clear')
submitBtnDelete.on_click(submitFunctionDelete)
clearBtnDelete.on_click(clear)
clearBtnDelete.linked_texts = [GameIDDelete]
successLabelDelete = widgets.Label(value = '')

deleteGroup = widgets.VBox(children = [deleteGame, GameIDDelete, EmptyLabel,
    submitBtnDelete, clearBtnDelete, successLabelDelete],
  layout = Layout(border = '3px solid white'))



In [ ]:
# @title gameSearch
def readFile(filename): #Returns contents of file as a list of lists
    f = open(f"{filename}","r")
    returnList = []
    for line in f:
        s = line.strip().split(",")
        returnList.append(s)
    f.close()
    return returnList

# will search through any and all records for a search term, then return list
#'all' is used for the initial state
def search(query, filename):
    databaseAsList = readFile(filename)
    resultList = []
    if query == 'all':
      return databaseAsList
    else:
      for lists in databaseAsList[1:]:
          for data in lists:
              if data.lower() == query.lower():
                  resultList.append(lists)
      return resultList

# The submit function for a search term
# Rebuilds table from the returned list
def gameSearch(btn):
  global videoTab, boardTab
  query = searchBox.value.strip()
  if query == '':
    return

  VideoTitles = (search('all', 'Video_Game_Info.txt'))[0]
  BoardTitles = (search('all', 'Board_Game_Info.txt'))[0]

  filteredV = [VideoTitles] + search(query, 'Video_Game_Info.txt')
  filteredB = [BoardTitles] + search(query, 'Board_Game_Info.txt')
  videoTableBox.children = [buildTable(filteredV)]
  boardTableBox.children = [buildTable(filteredB)]

title = widgets.Label(value = 'Find a game')
searchBox = widgets.Text(description = 'Search:',layout = Layout(width = '300'))
submitSearch = widgets.Button(description = 'Submit')
submitSearch.on_click(gameSearch)

searchGroup = [title,searchBox,submitSearch]

In [ ]:
# @title Subscription group
def submitFunctionSubscriptions(btn):
  ID = CustomerIDSubscriptions.value
  ID = ID.strip().lower()
  subsType = subscriptionType.value
  endDate = endDateUpdateSubs.value

  if ID == '' or endDate == None:
    successLabelSubscriptions.value = 'Please fill in all fields '
    return

  subscriptions = smSL.load_subscriptions()
  if ID in subscriptions:
    subscriptions[ID]['EndDate'] = endDate
    subscriptions[ID]['SubscriptionType'] = subsType
    updateFile = open('Subscription_Info.txt', 'w')
    # writing titles
    updateFile.write('CustomerID,SubscriptionType,StartDate,EndDate\n')

    for ID in subscriptions:
      subType = subscriptions[ID]['SubscriptionType']

      #Getting the date retrieves yyyy-mm-dd hh-mm-ss
      # so slicing for the first 10 characters gets rid of 'hh-mm-ss'
      startDate = str(subscriptions[ID]['StartDate'])[:10]
      endDate = str(subscriptions[ID]['EndDate'])[:10]
      line = f'{ID},{subType},{startDate},{endDate}\n'
      updateFile.write(line)

    updateFile.close()
    successLabelSubscriptions.value = 'Successfully updated user information'
    return

  else:
    successLabelSubscriptions.value = f'Could not find user ID: {ID}'

#Subscription widgets
updateSubcriptions = widgets.Label(value = 'Update Subscriptions')
CustomerIDSubscriptions = widgets.Text(description = 'Customer ID:',
                                       layout = Layout(width = '150px'))
subscriptionType = widgets.Dropdown(description = 'Subscription Type:',
                                    options = ['Standard', 'Premium'],
                                    layout = Layout(width = '200px'))
endDateUpdateSubs = widgets.DatePicker(description = 'End Date',
                                       layout = Layout(width = '200px'))
submitBtnSubscriptions = widgets.Button(description = 'Submit')
clearBtnSubscriptions = widgets.Button(description = 'Clear')
clearBtnSubscriptions.linked_texts = [CustomerIDSubscriptions]
submitBtnSubscriptions.on_click(submitFunctionSubscriptions)
successLabelSubscriptions = widgets.Label(value = '')
clearBtnSubscriptions.on_click(clear)

#Seperate but related section to subscriptions
def addNewSubscription(btn):
  ID = newCustomerID.value.strip()
  subType = newSubscriptionType.value
  startDate = startDateNewSubs.value
  endDate = endDateNewSubs.value

  if ID == '' or startDate == 'None' or endDate == None:
    successLabelSubscriptionsNew.value = 'Please fill in all fields'
    return

  #check not already in database
  current = search('all', 'Subscription_Info.txt')
  data = current[1:]

  for row in data:
    if row[0] == ID:
      successLabelSubscriptionsNew.value = 'Customer already exists'
      return

  #Too long of text lines made a database error, even with \
  #This is why there are two f"" below
  subsFile = open('Subscription_Info.txt', 'a')
  subsFile.write(f"{newCustomerID.value},{newSubscriptionType.value},"
  f"{startDateNewSubs.value},{endDateNewSubs.value}" + "\n")
  subsFile.close()
  successLabelSubscriptionsNew.value = 'Customer Successfully added'


#Subscription widgets
addNewSubscriptionLabel = widgets.Label(value = 'Add new subscription')
newCustomerID = widgets.Text(description = 'Customer ID')
newSubscriptionType = widgets.Dropdown(description = 'New Subscription Type:',
                                       options = ['Standard', 'Premium'],
                                       layout = Layout(width = '200px'))
submitNewSubscription = widgets.Button(description = 'Add')
submitNewSubscription.on_click(addNewSubscription)
startDateNewSubs = widgets.DatePicker(description = 'Start Date',
                                      layout = Layout(width = '200px'))
endDateNewSubs = widgets.DatePicker(description = 'End Date',
                                    layout = Layout(width = '200px'))

clearBtnNewSubscriptions = widgets.Button(description = 'Clear')
clearBtnNewSubscriptions.linked_texts = [newCustomerID]
clearBtnNewSubscriptions.on_click(clear)
successLabelSubscriptionsNew = widgets.Label(value = '')

#Group them all together for easy call in menu()
subscriptionGroup = [updateSubcriptions, CustomerIDSubscriptions,
                     subscriptionType, endDateUpdateSubs,
                     submitBtnSubscriptions, clearBtnSubscriptions,
                     successLabelSubscriptions, addNewSubscriptionLabel,
                     newCustomerID, newSubscriptionType, startDateNewSubs,
                     endDateNewSubs, submitNewSubscription,
                     clearBtnNewSubscriptions, successLabelSubscriptionsNew]


In [ ]:
# @title gameReturn

def gameReturn(btn = None):
  global screenBox
  '''
  The only type of nested function definitions you will see are for button
  pushes. This is because defining this button outside of the gameReturn means
  that the data contained inside gameReturn is no longer within
  submitReturnFunc()'s scope. it wont be able to access the widgets values
  such as - gameID = GameIDReturn.value - without using global variables. So
  I kept buttons as nested definitions.
  '''
  def submitReturnFunc(btn):
    gameID = GameIDReturn.value
    userID = customerIDReturn.value
    dateReturn = returnDate.value
    rating = ratingInt.value
    comment = commentText.value
    # Check first if this renting exists or not
    initialRead = readFile('Rental.txt')

    if gameID == '' or userID == '' or dateReturn == None:
      successLabelReturn.value = 'Fill in all Required fields'
      return

    subscriptions = smSL.load_subscriptions()
    if userID not in subscriptions:
      successLabelReturn.value = f"Could not find customer {userID}"
      return
    else:
      currentRfile = readFile('Rental.txt')
      newFile = []
      for rent in currentRfile:
        if rent[0] == gameID and rent[3] == userID:
          if rent[2] == '':
            rent[2] = str(dateReturn)
            successLabelReturn.value = "Saved return info"
            newFile.append(rent)

          else:
            successLabelReturn.value = "Already Returned"
            newFile.append(rent)
        else:
          newFile.append(rent)

      if successLabelReturn.value == "Already Returned":
        return # so this is here

      if newFile == initialRead:
        successLabelReturn.value = "Record not found!"
        return

      else:
        Rfile = open("Rental.txt", "w")
        for line in newFile:
          Rfile.write(f"{line[0]},{line[1]},{line[2]},{line[3]}\n")
        Rfile.close()
        successLabelReturn.value = "Saved return info"
        changeAvaliability(gameID, 1)

        #Now begin adding rating
        if rating == None or comment == '':
          #customer decided not to add rating or comment
          return

        else:
          fmSL.add_feedback(gameID, rating, comment, "Game_Feedback.txt")
          currentPopFile = readFile('Popularity.txt')
          titles = currentPopFile[0]
          Pfile = open('Popularity.txt', 'w')

          #increment total number of ratings
          newPopFile = [titles]
          for game in currentPopFile[1:]:
            rewrittenRecord = [game[0],game[1],game[2],game[3],game[4]]
            if game[0] == gameID:
              #increment number of ratings
              totalsCurrent = int(game[2])
              newTotal = totalsCurrent + 1
              rewrittenRecord[2] = newTotal

              # fix avg rating
              avgCurrent = float(game[1])
              sumRatings = avgCurrent * totalsCurrent
              sumRatings += int(rating)
              newAvg = sumRatings/int(totalsCurrent + 1)
              newAvg = int(newAvg * 100) / 100 #truncate
              rewrittenRecord[1] = newAvg
              newPopFile.append(rewrittenRecord)
            else:
              newPopFile.append(rewrittenRecord)

          for game in newPopFile:
            Pfile.write(f"{game[0]},{game[1]},{game[2]},{game[3]},{game[4]}\n")
          Pfile.close()
          rentals = readFile('Rental.txt')
          refreshedTable = buildTable(rentals)
          tableBoxRent.children = [refreshedTable]
          return

  # Define return widgets
  returnLabel = widgets.Label(value = 'Return Game')
  backButtonReturn = widgets.Button(description = 'Back')
  backButtonReturn.on_click(refresh)
  customerIDReturn = widgets.Text(description = 'Customer ID',
                                  layout = Layout(width = '150px'))
  GameIDReturn = widgets.Text(description = 'Game ID',
                              layout = Layout(width = '150px'))
  returnDate = widgets.DatePicker(description = 'Return Date',
                                  layout = Layout(width = '200px'))
  submitReturn = widgets.Button(description = 'Return')
  submitReturn.on_click(submitReturnFunc)
  optionalLabel = widgets.Label(value = 'Optional Rating')
  ratingInt = widgets.BoundedIntText(description = 'Rating', min = 0, max = 5,
                                  layout = Layout(width = '200px'))
  commentText = widgets.Text(description = 'Comment',
                                  layout = Layout(width = '400px'))

  #Rent here refers to rentals and not 'to rent a game'
  successLabelReturn = widgets.Label(value = '')
  rentTableTitle = widgets.Label(value = 'Rentals Database')
  rentTableTitleBox = widgets.VBox(children = [rentTableTitle],
                                   layout = Layout(height = '100px'))
  rentals = search('all', 'Rental.txt')
  tableRent = buildTable(rentals)
  tableBoxRent = widgets.VBox(children = [tableRent])
  leftRentChildren = [returnLabel, backButtonReturn, customerIDReturn,
                       GameIDReturn, returnDate,optionalLabel, ratingInt,
                       commentText,submitReturn, successLabelReturn]
  rightRentChildren = [rentTableTitleBox, tableBoxRent]

  leftRentBox = widgets.VBox(children = leftRentChildren,
                             layout = Layout(border = '2px solid white'))
  rightRentBox = widgets.VBox(children = rightRentChildren)
  screenBoxRent = widgets.HBox(children = [leftRentBox, rightRentBox],
                               layout = Layout(height = '600px',
                                               border = '2px solid white'))

  screenBox.children = [screenBoxRent]

In [ ]:
# @title feedbackBtnFunction

# Seperate window to view comments and ratings
def feedbackBtnFunction(Btn = None):
  global screenBox
  titleFeed = widgets.Label(value = 'Feedback')

  backBtnFeed = widgets.Button(description = 'Back')
  backBtnFeed.on_click(refresh)
  titleBox = widgets.HBox(children = [titleFeed,backBtnFeed],
                          layout = Layout(height = '100px'))
  feedback = fmSL.load_feedback()
  converted = feedbackToTableData(feedback)
  feedbackTable = buildTable(converted)
  feedbackTableBox = widgets.VBox(children = [feedbackTable],
                                  layout = Layout(border = '2px solid white'))
  screenBoxFeedback = widgets.VBox(children = [titleBox, feedbackTableBox],
                               layout = Layout(height = '600px',
                                               border = '2px solid white'))
  screenBox.children = [screenBoxFeedback]



In [ ]:
# @title feedbackToTableData

# Converts fmSL dictionary into a list the rest of my functions can use
def feedbackToTableData(feedback):
  headers = feedback[0].keys()
  tableData = []
  headerRow = []
  for title in headers:
    headerRow.append(title)
  tableData.append(headerRow)

  for entry in feedback:
    row = []
    for data in headers:
        row.append(str(entry[data]))
    tableData.append(row)

  return tableData

In [ ]:
# @title subscriptionsBtnFunction

# Window with table to view all subscriptions
def subscriptionsBtnFunction(Btn):

  titleSubs = widgets.Label(value = 'Subscription Info')

  backBtnSubs = widgets.Button(description = 'Back')
  backBtnSubs.on_click(refresh)
  titleBox = widgets.HBox(children = [titleSubs, backBtnSubs],
                          layout = Layout(height = '100px'))
  subs = readFile('Subscription_Info.txt')
  subsTable = buildTable(subs)
  subsTableBox = widgets.VBox(children = [subsTable])
  screenBoxSubs = widgets.VBox(children = [titleBox, subsTableBox],
                               layout = Layout(height = '600px',
                                               border = '2px solid white'))
  screenBox.children = [screenBoxSubs]



In [ ]:
# @title Remove group
# Remove qty from database - could be used for lost or faulty games
def submitFunctionRemove(btn):
  GameID = GameIDRemove.value
  qtyToRemove = int(quantityRemove.value)
  done = False

  x = changeAvaliability(GameID, -(qtyToRemove))

  if x[0] == False:
    successLabelRemove.value = f'{x[1]}: {GameIDRemove.value}'
  else:
    successLabelRemove.value = f'Removed {qtyToRemove} {GameIDRemove.value}'
  refreshDatabase()
  return

# define widgets for remove group
removeGame = widgets.Label(value = 'Remove faulty game')
GameIDRemove = widgets.Text(description = 'Game ID:',
                            layout = Layout(width = '150px'))
quantityRemove = widgets.BoundedIntText(description = 'Quantity', min = 1,
                                        layout = Layout(width = '150px'))

#placeholder for alignment
EmptyLabel = widgets.Label(value = '', layout = Layout(width = '150px'))
submitBtnRemove = widgets.Button(description = 'Remove')
clearBtnRemove = widgets.Button(description = 'Clear')
submitBtnRemove.on_click(submitFunctionRemove)
clearBtnRemove.on_click(clear)
clearBtnRemove.linked_texts = [GameIDRemove]
successLabelRemove = widgets.Label(value = '')

removeGroup = widgets.VBox(children = [removeGame, GameIDRemove, quantityRemove,
                                       EmptyLabel,submitBtnRemove,
                                       clearBtnRemove, successLabelRemove],
                           layout = Layout(border = '3px solid white'))



In [ ]:
# @title gameRent
def submitFunctionRent(btn):

  # Attempt to rent a game
  gameID = GameIDRent.value
  userID = CustomerIDRent.value
  today = dateToday.value

  done = False
  if gameID == '' or userID == '' or today == None:
    ErrorLabelRent.value = 'Please Fill in all fields'
    return

  #check user exists
  subscriptions = smSL.load_subscriptions()
  if userID in subscriptions:
    #check account active
    if smSL.check_subscription(userID, subscriptions) == True:
      userSubType = subscriptions[userID]['SubscriptionType']
      rentalLimit = smSL.get_rental_limit(userSubType)

      #now check how many they have currently rented out
      Rfile = readFile('Rental.txt')
      RfileData = Rfile[1:]#ignore titles

      rentalHistory = []
      for row in RfileData:
        if row[3] == userID:
          rentalHistory.append(row)
      unreturned = 0
      for rent in rentalHistory:
        if rent[2] == '' or None:
          unreturned += 1
      if unreturned >= rentalLimit:
        ErrorLabelRent.value = 'Customer has reached maximum limit'
        return
      else:
        #check in stock
        quantityAvaliable = 0
        boardGames = search('all', 'Board_Game_Info.txt')
        for item in boardGames:
          if item[0] == gameID:
            quantityAvaliable = item[-1]

        if quantityAvaliable == 0:
          videoGames = search('all', 'Video_Game_Info.txt')
          for item in videoGames:
            if item[0] == gameID:
              quantityAvaliable = item[-1]

        if quantityAvaliable == 0:
          ErrorLabelRent.value = 'Game not avaliable'
        else:
          # Game can be rented
          success = changeAvaliability(GameIDRent.value,-1)
          if success[0] == False:
            ErrorLabelRent.value = success[1]
            return


          Rfile = open('Rental.txt', 'a')
          Rfile.write(f"{gameID},{today},,{userID}\n")
          Rfile.close()

          #Update this months total rents for the game
          currentP = search('all', 'Popularity.txt')
          header = currentP[0]
          months = header[4].split(';')

          newMonthAdded = False
          #str to format time
          currentMonth = today.strftime('%d-%m-%Y')

          currentMonth = f'01-{currentMonth[3:5]}-{currentMonth[6:]}'

          if currentMonth not in months:
            months.append(currentMonth)
            header[4] = ';'.join(months)
            newMonthAdded = True

          if newMonthAdded == True:
            for game in currentP[1:]:
              history = game[4].split(';')
              history.append('0')
              game[4] = ';'.join(history)

          newPopFile = []
          for game in currentP:
            if game[0] == gameID:
              rentHistory = game[-1].split(';')
              if newMonthAdded == True:
                last = int(rentHistory[-2])
              else:
                last = int(rentHistory[-1])
              last += 1
              last = str(last)
              rentHistory[-1] = last
              newHistory = ';'.join(rentHistory)
              game[-1] = newHistory

              #Total rents also increments
              totalsCurrent = int(game[3])
              newTotal = str(totalsCurrent + 1)
              game[3] = newTotal

              newPopFile.append(game)
            else:
              newPopFile.append(game)

          Pfile = open('Popularity.txt', 'w')
          for row in newPopFile:
            Pfile.write(','.join(row) + '\n')
          Pfile.close()
          ErrorLabelRent.value = 'Rented Game'
    else:
      ErrorLabelRent.value = 'Account not active'

  else:
    ErrorLabelRent.value = 'Could not complete'

# define rent groups widgets
rentGame = widgets.Label(value = 'Rent a Game')
GameIDRent = widgets.Text(description = 'Game ID:',
                          layout = Layout(width = '150px'))
CustomerIDRent = widgets.Text(description = 'Customer ID:',
                              layout = Layout(width = '150px'))
dateToday = widgets.DatePicker(description = 'Date of Rent',
                               layout = Layout(width = '200px'))

submitBtnRent = widgets.Button(description = 'Submit')
clearBtnRent = widgets.Button(description = 'Clear')
clearBtnRent.linked_texts = [GameIDRent, CustomerIDRent]
submitBtnRent.on_click(submitFunctionRent)
clearBtnRent.on_click(clear)
ErrorLabelRent = widgets.Label(value = '')

rentGroup = widgets.VBox(children = [rentGame, GameIDRent, CustomerIDRent,
                                     dateToday, submitBtnRent, clearBtnRent,
                                     ErrorLabelRent],
                          layout = Layout(border = '3px solid white'))

In [ ]:
# @title Add group
def submitFunctionAdd(btn):
    radio = widgets.RadioButtons(description = 'Game Type:',
                                 options = ['Video Game', 'Board Game'])
    def updateGameType(change):
      #changes widget depending on game type
      if change['new'] == 'Video Game':
          gameTypeDisplay.children = [videoDisplay]
      else:
          gameTypeDisplay.children = [boardDisplay]

    radio.observe(updateGameType, names = 'value')

    def submitVFunction(btn):
      #if the radio button is set to video games
      f = open('Video_Game_Info.txt','a')

      f.write(ID.value.strip() + ",")
      f.write(name.value.strip() + ",")
      f.write(multiplayer.value.strip() + ",")
      f.write(genre.value.strip() + ",")
      f.write(platform.value.strip() + ",")
      f.write(str(purchaseDate.value) + ",")
      f.write(quantity.value.strip() + "\n")

      f.close()
      v = open('Popularity.txt', 'r')
      line = v.readline()
      listLine = line.split(',')
      listLine = listLine[-1]
      listLine = listLine.split(';')
      nMonths = len(listLine)
      lastColumnNew = '0;' * nMonths
      lastColumnNew = lastColumnNew[:-1]
      v.close()

      v = open('Popularity.txt', 'a')
      v.write(f"{ID.value.strip()},0,0,0," + lastColumnNew)
      v.close()
      successLabel.value = 'Successfully added'

    def submitBFunction(btn):
      #if the radio button is set to board games
      f = open('Board_Game_Info.txt','a')
      f.write(ID.value.strip() + ",")
      f.write(name.value.strip() + ",")
      f.write(players.value.strip() + ",")
      f.write(genre.value.strip() + ",")
      f.write(str(purchaseDate.value) + ",")
      f.write(quantity.value.strip() + "\n")
      f.close()

      b = open('Popularity.txt', 'r')
      line = b.readline()
      listLine = line.split(',')
      listLine = listLine[-1]
      listLine = listLine.split(';')
      nMonths = len(listLine)
      lastColumnNew = '0;' * nMonths
      lastColumnNew = lastColumnNew[:-1]
      b.close()

      b = open('Popularity.txt', 'a')
      b.write(f"{ID.value.strip()},0,0,0," + lastColumnNew + '\n')
      b.close()
      successLabel.value = 'Successfully added'


    # define widgets for Add new group
    title = widgets.Label(value = "Add new game")


    ID = widgets.Text(description = 'Game ID',
                      layout = Layout(width = '150px'))
    name = widgets.Text(description = 'Game Name',
                        layout = Layout(width = '150px'))
    multiplayer = widgets.Dropdown(description = 'Multiplayer',
                                   options = ['Multiplayer', 'Solo'],
                                   layout = Layout(width = '200px'))
    players = widgets.Text(description = 'Number of players',
                           layout = Layout(width = '150px'))
    genre = widgets.Text(description = 'Genre',
                         layout = Layout(width = '150px'))
    platform = widgets.Text(description = 'Platform',
                            layout = Layout(width = '150px'))
    purchaseDate = widgets.DatePicker(description = 'Purchase Date',
                                      layout = Layout(width = '200px',))
    quantity = widgets.Text(description = 'Quantity',
                            layout = Layout(width = '150px'))

    submitVButton = widgets.Button(description = 'Submit')
    submitVButton.on_click(submitVFunction)
    successLabel = widgets.Label(value = '')
    submitBButton = widgets.Button(description = 'Submit')
    submitBButton.on_click(submitBFunction)
    backButton = widgets.Button(description = 'Back')
    backButton.on_click(refresh)

    videoDisplay = widgets.VBox([ID, name, multiplayer, genre,
        platform, purchaseDate, quantity, submitVButton])

    boardDisplay = widgets.VBox([ID, name, players, genre,
        purchaseDate, quantity,submitBButton])
    gameTypeDisplay = widgets.VBox()
    gameTypeDisplay.children = [videoDisplay]

    finalDisplay = widgets.VBox(children = [title, backButton, radio,
                                            gameTypeDisplay, successLabel])
    global screenBox
    screenBox.children = [finalDisplay]

In [ ]:
# @title Add copy group
def addCopyFunction(btn):
  GameID = GameIDAddCopy.value
  qtyToAdd = quantityAddCopy.value
  success = changeAvaliability(GameID, qtyToAdd)

  if success[0] == False:
    successLabelAddCopy.value = f'{success[1]}: {GameID}'
  else:
    successLabelAddCopy.value = f'Added {qtyToAdd} copies to {GameID}'
  refreshDatabase()
  return

# define add copy widgets
addLabel = widgets.Label(value = 'Add a new game')
AddNewGame = widgets.Button(description = 'Add new game')
addCopyLabel = widgets.Label(value = 'Add a new copy')
GameIDAddCopy = widgets.Text(description = 'Game ID',
                             layout = Layout(width = '150px'))
quantityAddCopy = widgets.BoundedIntText(description = 'Quantity', min = 1,
                                         layout = Layout(width = '150px'))
successLabelAddCopy = widgets.Label(value = '')
addCopyButton = widgets.Button(description = 'Add copy')
addCopyButton.on_click(addCopyFunction)
AddNewGame.on_click(submitFunctionAdd)

addGroup = widgets.VBox(children = [addLabel, AddNewGame, addCopyLabel,
                                    GameIDAddCopy, quantityAddCopy,
                                    addCopyButton, successLabelAddCopy ],
                        layout = Layout(border = '3px solid white'))

In [ ]:
# @title Book session group
def bookSessionFunc(btn = None):

  def submitBookingFunction(btn):
    userID = customerIDBook.value
    time = timeSlot.value
    day = bookingDate.value
    guestNumber = numGuests.value#]
    print(day)

    if userID == '' or day == None:
      successLabelBooking.value = 'Please fill in all fields'
      return
    #add checks for that day
    subscriptions = smSL.load_subscriptions()
    activeUser = smSL.check_subscription(userID, subscriptions)

    if userID not in subscriptions:
      successLabelBooking.value = 'Customer Not found'
      return
    elif activeUser == False:
      successLabelBooking.value = 'Account not active'
      return

    #Check if space (<50)
    currentHistory = readFile('Booking.txt')
    currentCapacity = 0
    for row in currentHistory[1:]:
      if row[1].strip() == str(day) and row[2].strip() ==time and len(row) == 4:
        currentCapacity += int(row[3]) + 1

    # 1 is the customer themselves
    newCapacity = currentCapacity + guestNumber + 1

    if newCapacity > 50:
      successLabelBooking.value = 'Not enough space for this session'
      return

    Bfile = open('Booking.txt', 'a')
    Bfile.write(f"{userID},{day},{time},{guestNumber}\n")
    Bfile.close()
    bookingFile = search('all', 'Booking.txt')
    updatedTable = buildTable(bookingFile)
    bookingTableBox.children = [updatedTable]
    successLabelBooking.value = 'Successfully booked session'

  # widgets for booking group
  bookLabel = widgets.Label(value = 'Book Session')
  backButtonBook = widgets.Button(description = 'Back')
  backButtonBook.on_click(refresh)
  customerIDBook = widgets.Text(description = 'Customer ID',
                                layout = Layout(width = '150px'))
  bookingDate = widgets.DatePicker(description = 'Date',
                                  layout = Layout(width = '200px'))
  timeSlot = widgets.Dropdown(description = 'Time slot',
                              options = ['2pm-6pm', '6pm-10pm'],
                              layout = Layout(width = '200px'))
  numGuests = widgets.BoundedIntText(description = 'Number of Guests',
                                    min = 0, max = 3,
                                    layout = Layout(width = '150px'))
  submitBooking = widgets.Button(description = 'Submit Booking')
  submitBooking.on_click(submitBookingFunction)
  successLabelBooking = widgets.Label(value = '')

  currentBookings = search('all', 'Booking.txt')
  bookingsTable = buildTable(currentBookings)
  bookingTableBox = widgets.VBox(children = [bookingsTable])
  tableHeader = widgets.Label(value = 'Bookings Table')
  bookTableHeaderBox = widgets.VBox(children = [tableHeader],
                                    layout = Layout(height = '100px'))
  leftBoxBooking = widgets.VBox(children = [bookLabel, backButtonBook,
                                            customerIDBook,bookingDate,
                                            timeSlot,numGuests,submitBooking,
                                            successLabelBooking],
                                layout = Layout(border = '2px solid white'))

  rightBoxBooking = widgets.VBox(children = [bookTableHeaderBox, bookingTableBox])
  bookingScreen = widgets.HBox(children = [leftBoxBooking, rightBoxBooking],
                               layout = Layout(border = '2px solid white',
                                               height = '700px'))

  global screenBox
  screenBox.children = [bookingScreen]

#Database
Here contains most of my functions relating to manipulation of the database, apart from the specified functions above

In [ ]:
# @title refreshDatabaseInventory
def refreshDatabaseInventory(btn = None):
    inventoryDB = buildTable(search('all', 'Popularity.txt'))
    popularityTable.children = [inventoryDB]

In [ ]:
# @title buildPopularityGraph
def buildPopularityGraph(gameID):
    months = getRentalHistory(gameID)[0]
    rents = getRentalHistory(gameID)[1]

    plt.close('all')
    fig, ax = plt.subplots(figsize = (5, 3))
    mIndex = 0

    while mIndex < len(months):
      month = months[mIndex]
      if month[3:5] == '01':
        months[mIndex] = 'Jan'
      elif month[3:5] == '02':
        months[mIndex] = 'Feb'
      elif month[3:5] == '03':
        months[mIndex] = 'Mar'
      elif month[3:5] == '04':
        months[mIndex] = 'Apr'
      elif month[3:5] == '05':
        months[mIndex] = 'May'
      elif month[3:5] == '06':
        months[mIndex] = 'Jun'
      elif month[3:5] == '07':
        months[mIndex] = 'Jul'
      elif month[3:5] == '08':
        months[mIndex] = 'Aug'
      elif month[3:5] == '09':
        months[mIndex] = 'Sep'
      elif month[3:5] == '10':
        months[mIndex] = 'Oct'
      elif month[3:5] == '11':
        months[mIndex] = 'Nov'
      elif month[3:5] == '12':
        months[mIndex] = 'Dec'

      mIndex += 1

    if len(months) > 0:
        ax.plot(months, rents, marker = 'o')
        ax.set_title(f"Rents Per Month ({gameID})")
        ax.set_xlabel('Month')
        ax.set_ylabel('Number of Rents')
    else:
        ax.set_title('No data available')

    plt.tight_layout() # automatically fits widget
    return(fig)


In [ ]:
# @title onSelected
def onSelect(btn):
    global selectedRow, rentalGraph

    # Reset previous selection then highlight new one
    if selectedRow != None:
        selectedRow.layout.border = ''

    btn.rowBox.layout.border = '3px solid blue'
    selectedRow = btn.rowBox # fetch data associated with button

    # Auto fill rows
    selectedID = btn.rowData[0]
    GameIDRent.value = selectedID
    GameIDAddCopy.value = selectedID
    GameIDRemove.value = selectedID
    GameIDDelete.value = selectedID


    # Purely a block for the bookings table
    # check if the ID is a userID

    subscriptions = smSL.load_subscriptions()
    if selectedID in subscriptions:
      return

    fig = buildPopularityGraph(selectedID)

    rentalGraph.clear_output(wait = True)
    with rentalGraph:
      display(fig)

    plt.close(fig)


In [ ]:
# @title inventoryPruning
def updatePopularityTable(change=None):
    database = search('all', 'Popularity.txt')
    sortedDB = sortPopularity(database, orderByDropdown.value)
    table = buildTable(sortedDB, popFile = True)
    popularityTable.children = [table]
def openInventory(btn):
  clear_output(wait=True)
  inventoryPruning()

def inventoryPruning(btn = None):

  #Window to see how popular games are with graph

  global orderByDropdown, popularityTable, rentalGraph, selectedRow

  titlePruning = widgets.Label(value = 'Inventory Pruning')
  backButton = widgets.Button(description = 'Back')
  backButton.on_click(refresh)
  topButtonsBox = widgets.HBox(children = [backButton])
  databaseHeader = widgets.Label(value = 'Game Usage')
  orderByDropdown = widgets.Dropdown(description = 'Order by:',
                                     options = ['GameID','Average Rating',
                                               'No. of Ratings','Total Rents',
                                               'Rents this month'],
                                     layout = Layout(width = '300px'))
  orderByDropdown.observe(updatePopularityTable, names = 'value')

  popularityFile = search('all', 'Popularity.txt')
  table = buildTable(popularityFile, popFile = True)
  popularityTable = widgets.VBox(children = [table],
                                 layout = Layout(max_height = '500px',
                                                 overflow_y = 'auto',
                                                 overflow_x = 'hidden',
                                                 border = '1px solid gray'))



  pruningLeftBoxChildren = [titlePruning, topButtonsBox,
                            orderByDropdown, popularityTable]
  title = widgets.Label(value = 'Graph of game usage')
  instructionLabel = widgets.Label(value = 'Select game to get started')
  global rentalGraph, screenBox
  rentalGraph = widgets.Output()

  pruningRightBoxChildren = [title, instructionLabel, rentalGraph]


  pruningLeftBox = widgets.VBox(layout = Layout(width = '720px',
                                                border = '2px solid white'),
                                children = pruningLeftBoxChildren)
  pruningRightBox = widgets.VBox(layout = Layout(width = '680px',
                                                 border = '2px solid white'),
                                 children = pruningRightBoxChildren)

  items = [pruningLeftBox,pruningRightBox]
  screenBoxI = widgets.HBox(children = items)

  updatePopularityTable()
  screenBox.children = [screenBoxI]


In [ ]:
# @title changeAvaliability
def changeAvaliability(GameID, quantity):
  currentV = search('all','Video_Game_Info.txt')
  currentB = search('all','Board_Game_Info.txt')
  quantity = int(quantity)
  vTitles = currentV[0]
  vData = currentV[1:]
  bTitles = currentB[0]
  bData = currentB[1:]

  newVidRows = []
  if quantity < 0:
    errorLabel = f'Removed {quantity} {GameID}'
    mode = 'Sub'

  else:
    errorLabel = f'Added {quantity} {GameID}'
    mode = 'Add'

  done = False

  #Check game exists
  exists = False
  for item in vData:
    if item[0] == GameID:
      exists = True
  for item in bData:
    if item[0] == GameID:
      exists = True
  if exists == False:
    errorLabel = 'Game not in Database'
    return(False, errorLabel)


  for item in vData:
    if item[0] == GameID and done == False:
      quantityCurrent = int(item[-1])
      if quantityCurrent <= 0 and mode == 'Sub':
        errorLabel = 'Game out of stock'
        return(done, errorLabel)

      quantityNew = quantityCurrent + quantity

      if quantityNew < 0 and mode == 'Sub':
        errorLabel = 'Not enough Copies avaliable'

      item[-1] = str(quantityNew)
      done = True

    newVidRows.append(item)

  if done == True:
    Vfile = open('Video_Game_Info.txt', 'w')
    stringTitlesV = ",".join(vTitles) + "\n"
    Vfile.write(stringTitlesV)
    for row in newVidRows:
      Vfile.write(','.join(row) + "\n")
    Vfile.close()
    refreshDatabase()
    return(done, errorLabel)



  newBoardRows = []

  for item in bData:
    if item[0] == GameID and done == False:
      quantityCurrent = int(item[-1])
      if quantityCurrent <= 0 and mode == 'Sub':
        errorLabel = 'Game fully rented out'
        return(done, errorLabel)
      quantityNew = quantityCurrent + quantity

      if quantityNew < 0 and mode == 'Sub':
        errorLabel = 'Not enough Copies avaliable'

      item[-1] = str(quantityNew)
      done = True

    newBoardRows.append(item)

  if done == True:
    Bfile = open('Board_Game_Info.txt', 'w')
    BTitles = ",".join(bTitles) + "\n"
    Bfile.write(BTitles)
    for row in newBoardRows:
      Bfile.write(','.join(row) + "\n")
    Bfile.close()

  return(done, errorLabel)

In [ ]:
# @title sortPopularity
def sortPopularity(data, orderBy):
  global sortColumn

  titles = data[0]
  data = data[1:]

  if orderBy == 'GameID':
    sortColumn = 0
  elif orderBy == 'Average Rating':
    sortColumn = 1
  elif orderBy == 'No. of Ratings':
    sortColumn = 2
  elif orderBy == 'Total Rents':
    sortColumn = 3
  else: # rents this month
    sortColumn = 4
#
  if sortColumn != 0:#Turns to numbers
    index = 0
    while index < len(data):
      if len(data[index]) <= sortColumn:
        continue

      if sortColumn != 1:
        monthRents = data[index][sortColumn].split(';')
        data[index][sortColumn] = int(monthRents[-1])
      else:
        data[index][sortColumn] = float(data[index][sortColumn])
      index += 1

  data.sort(key = getSortValue, reverse = True)

  if sortColumn != 0:#Turn back to strings
    index = 0
    while index < len(data):
      data[index][sortColumn] = str(data[index][sortColumn])
      index += 1

  sortedDatabasePruning = [titles]
  for row in data:
    sortedDatabasePruning.append(row)
  return sortedDatabasePruning

#helper function
def getSortValue(row): # which column will the .sort() function order by
  return row[sortColumn]

In [ ]:
# @title refresh
def refresh(btn):
  refreshDatabase()
  clear_output()
  menu()

In [ ]:
# @title buildTable
def buildTable(data, popFile = False):

    # Constructs a table in the form:
    # every row is one GridBox containing Label widgets
    # Label widgets filled with data from database
    # full table is a VBox that stacks each GridBox row

    global videoTab, boardTab
    noData = widgets.Label(value = 'No data found')
    titles = data[0]

    # Popularity.txt is different and i only want to show a specifc
    # part of the last column, so it needs it own section
    if popFile == True:
      lastColumn = 'Rents this month'
      months = titles[-1].split(';')
      titles = titles[:-1]  # keep most recent month
      titles.append(lastColumn)

    titleWidgets = [widgets.Label(value = 'Select')]
    for t in titles:
      titleWidgets.append(widgets.Label(value = t))



    if len(data[0]) == 3:
      columns = '90px 120px 120px 400px'
    else:
      columns = '90px ' + '120px ' * len(titles)
    header = widgets.GridBox(children = titleWidgets,
                             layout = Layout(
                      grid_template_columns = columns))

    if len(data) == 1:
      return widgets.VBox(children = [header, noData])

    tableRows = [header]
    global currentTableRows
    currentTableRows = []

    rows = data[1:]
    for row in rows:
      if popFile == True:
        lastColumn = row[-1]
        months = lastColumn.split(';')
        row = row[:-1]
        row.append(months[-1])# keep most recent month


      rowData = []
      selectBtn = widgets.Button(description = 'Select',
                                  layout = Layout(width = '80px'))
      selectBtn.rowData = row  # store full row data to button
      rowData.append(selectBtn)

      for cell in row:
        rowData.append(widgets.Label(value = cell))
      rowBox = widgets.GridBox(children = rowData, layout = Layout(
              grid_template_columns = columns))

      selectBtn.rowBox = rowBox  # link button to its rowBox
      selectBtn.on_click(onSelect)

      tableRows.append(rowBox)
      currentTableRows.append(rowBox)

    return widgets.VBox(children = tableRows,
                        layout = Layout(display = 'block'))

In [ ]:
# @title getRentalHistory
def getRentalHistory(gameID):
  data = search('all', 'Popularity.txt')
  header = data[0]
  monthTitles = header[-1].split(';')

  for row in data[1:]:
      if row[0] == gameID:
          rentValues = row[-1].split(';')

          # keep last 6 months
          monthTitles = monthTitles[-6:]
          rentValues = rentValues[-6:]

          # convert rents to int
          rentCount = []
          for r in rentValues:
              rentCount.append(int(r))
          # could be a new game with no history
          if len(monthTitles) != 0 and len(rentCount) != 0:
            return (monthTitles, rentCount)
          else:
            return (0,0)
  return (0,0)




In [ ]:
# @title refreshDatabase
def refreshDatabase(btn = None):
    videoDB = buildTable(search('all', 'Video_Game_Info.txt'))
    boardDB = buildTable(search('all', 'Board_Game_Info.txt'))
    videoTableBox.children = [videoDB]
    boardTableBox.children = [boardDB]


In [ ]:
# @title menu
def menu():

  # main widgets
  title = widgets.Label(value = "Game Management System")
  customersLabel = widgets.Label(value = 'Customer Management')
  refreshButton = widgets.Button(description = "Refresh")
  refreshButton.on_click(refreshDatabase)
  pruneButton = widgets.Button(description = 'Inventory Pruning')
  pruneButton.on_click(openInventory)

  returnGameBtn = widgets.Button(description = 'Open return Game')
  returnGameBtn.on_click(gameReturn)
  bookSessionBtn = widgets.Button(description = 'Book new session')
  bookSessionBtn.on_click(bookSessionFunc)
  subscriptionsBtn = widgets.Button(description = 'View subscriptions')
  subscriptionsBtn.on_click(subscriptionsBtnFunction)
  feedbackBtn = widgets.Button(description = 'View feedback')
  feedbackBtn.on_click(feedbackBtnFunction)

  topButtonsBox = widgets.HBox(children = [refreshButton, pruneButton,
                                           returnGameBtn, bookSessionBtn,
                                           subscriptionsBtn, feedbackBtn])


  #Main screen tab for video games and board games
  tabHeader = widgets.Label(value = 'Database')


  tabDatabases = []
  global selectedRow, screenBox, videoTableBox, boardTableBox
  videoTableBox = widgets.VBox(
      children = [buildTable(search('all', 'Video_Game_Info.txt'))])
  vidTabTitle = widgets.Label(value = 'Video games')
  videoTab = widgets.VBox(children = [vidTabTitle, videoTableBox])
  boardTableBox = widgets.VBox(
      children = [buildTable(search('all', 'Board_Game_Info.txt'))])
  boardTabTitle = widgets.Label(value = 'Board games')
  boardTab = widgets.VBox(children = [boardTabTitle, boardTableBox])

  selectedRow = None

  currentTableRows = []
  tab = widgets.Tab(titles = ['Video games', 'Board Games'],
                    layout = Layout(width = '100%', height = '450px'),
                    children = [videoTab,boardTab])


  LBoxChildren = [title, customersLabel] + subscriptionGroup
  DBoperationsList = [rentGroup, addGroup, removeGroup, deleteGroup]
  DatabaseOperations = widgets.HBox(children = DBoperationsList)
  RBoxChildren = [tabHeader, topButtonsBox] + searchGroup + [tab,
                                                             DatabaseOperations]

  leftBox = widgets.VBox(layout = Layout(width = '350px',
                                         border = '2px solid white'),
                         children = LBoxChildren)
  rightBox = widgets.VBox(layout = Layout(width = '1000px',
                                          border = '2px solid white'),
                          children = RBoxChildren)
  items = [leftBox,rightBox]

  screenBox = widgets.HBox(children = items)

  display(screenBox)



#End
The end of the notebook. Run the cell below to see it in action.


 When running the notebook for the first time it is imperative you run inventoryPruning first! This is because when a button is selected BEFORE opening inventoryPruning, a graph is instantiated with nowhere to be displayed (yet).
This bug was found in final testing. When i had it up and running, Colab remembered from previous runs that the graph exists in inventory pruning, but resetting the system forgets this information!

In [ ]:
#Run this first
inventoryPruning()

In [ ]:
menu()

#5 security code issues
1) No restricted Access to the system. I have not included password setting functionality, so if the system is left unattended (i.e manager goes on break) the code is vulnerable
2) No encryption. All information is stored in a flatfile. When being accessed or transmitted over the internet, data is vulnerable.
3) Likely prone to injection or some similar attack. All text entries cells have not got proper malicious code preventions built in.
4) System load bearing has not been tested. In the event of continual use, the system may not eventually be able to handle the strain and crash, posing a risk of a loss of data.
5) No log type structure. If code is manipulated maliciously (i.e a customer extends their subscription infinitely) the system won't alert anyone of the change and could go unnoticed.

Statement on the use of GenAI Tools

Generative AI tools were used in a limited, assistive way during this coursework. They were mainly used to help with understanding Python syntax, fixing errors in code, and proofreading written sections.

All program logic, final code, and decisions about how the system works were my own. Any suggestions from AI tools were reviewed and adapted by me, and no complete code solutions or result interpretations were generated by AI.